# Lesson 9.3 - Advanced NLP & LLM Applications (seq2seq/LLM demo)

## Objectives

- Run seq2seq summarization with a small transformer model.
- Demonstrate structured output prompting.
- Connect outputs to evaluation and safety concerns.


In [1]:
from __future__ import annotations

from transformers import pipeline
import json


## Summarization with Seq2Seq Model

This demo uses a compact summarization model to keep runtime manageable.


In [2]:
generator = pipeline("text-generation", model="distilgpt2", device=-1)

text = (
    "Reinforcement learning is a machine learning paradigm where an agent learns to make decisions "
    "by interacting with an environment. The agent receives rewards and aims to maximize cumulative "
    "return over time. Practical RL often struggles with sample inefficiency and training instability, "
    "which motivates replay buffers, target networks, and robust evaluation protocols."
)

prompt = f"Summarize this in 2 concise sentences:\n\n{text}\n\nSummary:"
summary = generator(
    prompt,
    max_new_tokens=40,
    do_sample=False,
    pad_token_id=generator.tokenizer.eos_token_id,
)[0]["generated_text"]
print(summary)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Summarize this in 2 concise sentences:

Reinforcement learning is a machine learning paradigm where an agent learns to make decisions by interacting with an environment. The agent receives rewards and aims to maximize cumulative return over time. Practical RL often struggles with sample inefficiency and training instability, which motivates replay buffers, target networks, and robust evaluation protocols.

Summary:
The first step in learning reinforcement learning is to learn how to apply reinforcement learning to a given situation. The second step is to learn how to apply reinforcement learning to a given situation.
The first


## Translation-Style Prompting with Text2Text


In [3]:
generator = pipeline("text-generation", model="distilgpt2", device=-1)

text = (
    "Reinforcement learning is a machine learning paradigm where an agent learns to make decisions "
    "by interacting with an environment. The agent receives rewards and aims to maximize cumulative "
    "return over time. Practical RL often struggles with sample inefficiency and training instability, "
    "which motivates replay buffers, target networks, and robust evaluation protocols."
)

prompt = f"Summarize this in 2 concise sentences:\n\n{text}\n\nSummary:"
summary = generator(
    prompt,
    max_new_tokens=40,
    do_sample=False,
    pad_token_id=generator.tokenizer.eos_token_id,
)[0]["generated_text"]
print(summary)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize this in 2 concise sentences:

Reinforcement learning is a machine learning paradigm where an agent learns to make decisions by interacting with an environment. The agent receives rewards and aims to maximize cumulative return over time. Practical RL often struggles with sample inefficiency and training instability, which motivates replay buffers, target networks, and robust evaluation protocols.

Summary:
The first step in learning reinforcement learning is to learn how to apply reinforcement learning to a given situation. The second step is to learn how to apply reinforcement learning to a given situation.
The first


## Structured Output Prompting (JSON Template)


In [4]:
analysis_prompt = (
    "Summarize this issue and output JSON with keys: risk, mitigation, owner. "
    "Issue: The chatbot hallucinates unsupported legal clauses in contract summaries."
)

structured_text = generator(
    analysis_prompt + "\nJSON:",
    max_new_tokens=80,
    do_sample=False,
    pad_token_id=generator.tokenizer.eos_token_id,
)[0]["generated_text"]
print(structured_text)

# Optional post-processing skeleton:
def try_parse_json(s: str):
    import json
    try:
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(s[start:end+1])
        return {"raw": s}
    except Exception:
        return {"raw": s}

try_parse_json(structured_text)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize this issue and output JSON with keys: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries.
JSON: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries. JSON: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries. JSON: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries. JSON: risk, mitigation, owner. Issue: The chatbot


{'raw': 'Summarize this issue and output JSON with keys: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries.\nJSON: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries. JSON: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries. JSON: risk, mitigation, owner. Issue: The chatbot hallucinates unsupported legal clauses in contract summaries. JSON: risk, mitigation, owner. Issue: The chatbot'}

## Connect to Theory

- Seq2seq summarization uses conditional generation from source text.
- Text2text models unify multiple tasks under prompt conditioning.
- Structured outputs increase integration reliability for downstream systems.


## Business Case Studies & Exceptions

### Case A: Enterprise Support Summarization

Teams use summarization to compress long ticket threads. Strong retrieval grounding and human QA are needed for policy-sensitive responses.

### Case B: Developer Copilot Workflows

Code assistants reduce boilerplate and speed reviews, but need security/static checks to prevent vulnerable suggestions.

### Exceptions

- For strict deterministic transformations, rule-based pipelines may beat LLMs.
- In small, well-labeled domains, compact supervised NLP models may be cheaper and more controllable.


## Interview Questions & Answers

1. **Q: What is seq2seq modeling?**
   **A:** Mapping one sequence to another using encoder-decoder architectures.
2. **Q: What is masked LM vs causal LM?**
   **A:** Masked predicts hidden tokens; causal predicts next token autoregressively.
3. **Q: Why use instruction tuning?**
   **A:** To improve general instruction-following behavior.
4. **Q: What is RLHF used for?**
   **A:** Aligning model outputs with human preferences and safety goals.
5. **Q: How do you evaluate summarization quality?**
   **A:** Automatic metrics + human factuality/utility checks.
6. **Q: Why is groundedness important?**
   **A:** It reduces hallucinated claims.
7. **Q: What is structured output?**
   **A:** Constraining generations to typed schemas like JSON.
8. **Q: How do you mitigate jailbreak risk?**
   **A:** Prompt policies, filtering, routing, and adversarial tests.
9. **Q: When use smaller models in production?**
   **A:** When latency/cost/privacy constraints are strict.
10. **Q: How would you productionize this demo?**
   **A:** Add retrieval, eval harness, monitoring, and fallback logic.
